# Train BG and SB L2O models

This notebook trains and evaluates the BG and SB surrogate models by directly calling `L2Otraining.py`.

Expected input files:
- `data/X_BG.npy` and `data/Y_BG.npy`
- `data/X_SB.npy` and `data/Y_SB.npy`

Model checkpoints and metadata are saved under `NNmodel/`.

In [1]:
from pathlib import Path

import torch

from L2Otraining import TrainingConfig, evaluate, load_data, train

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "L2Otraining.py").is_file():
    raise FileNotFoundError(
        "Please open this notebook from the project root containing L2Otraining.py."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Training device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Project root: /Users/liqi/Library/CloudStorage/OneDrive-TheUniversityofHongKong-Connect/Stackelberg Game/Surrogated-assisted Stackelnerg Game
Training device: cpu


## Training configuration

Adjust the shared hyperparameters below if needed. Separate `TrainingConfig` objects are created for BG and SB so they can be changed independently.

In [2]:
ALL_SCENARIOS = ("SB1", "SB2", "BG1", "BG2", "IP1",)
SCENARIOS = ("SB1", "SB2", "BG1", "BG2", "IP1",)

configs = {
    scenario: TrainingConfig(
        scenario=scenario,
        hidden_dim=128,
        residual_blocks=3,
        batch_size=64,
        learning_rate=1e-3,
        epochs=500,
        validation_fraction=0.2,
        seed=42,
    )
    for scenario in SCENARIOS
}

configs

{'SB1': TrainingConfig(scenario='SB1', hidden_dim=128, residual_blocks=3, batch_size=64, learning_rate=0.001, epochs=500, validation_fraction=0.2, seed=42),
 'SB2': TrainingConfig(scenario='SB2', hidden_dim=128, residual_blocks=3, batch_size=64, learning_rate=0.001, epochs=500, validation_fraction=0.2, seed=42),
 'BG1': TrainingConfig(scenario='BG1', hidden_dim=128, residual_blocks=3, batch_size=64, learning_rate=0.001, epochs=500, validation_fraction=0.2, seed=42),
 'BG2': TrainingConfig(scenario='BG2', hidden_dim=128, residual_blocks=3, batch_size=64, learning_rate=0.001, epochs=500, validation_fraction=0.2, seed=42),
 'IP1': TrainingConfig(scenario='IP1', hidden_dim=128, residual_blocks=3, batch_size=64, learning_rate=0.001, epochs=500, validation_fraction=0.2, seed=42)}

## Check training data

This verifies that all four arrays exist and that each input/output pair has aligned sample counts before starting the longer training run.

In [3]:
data_shapes = {}
for scenario in SCENARIOS:
    x_tensor, y_tensor = load_data(scenario)
    data_shapes[scenario] = {
        "X shape": tuple(x_tensor.shape),
        "Y shape": tuple(y_tensor.shape),
    }

data_shapes

{'SB1': {'X shape': (10000, 24), 'Y shape': (10000, 24)},
 'SB2': {'X shape': (10000, 24), 'Y shape': (10000, 24)},
 'BG1': {'X shape': (10000, 24), 'Y shape': (10000, 24)},
 'BG2': {'X shape': (10000, 24), 'Y shape': (10000, 24)},
 'IP1': {'X shape': (10000, 24), 'Y shape': (10000, 24)}}

## Train both models

The existing `train` function handles the train/validation split, optimization, progress output, checkpoint saving, and metadata saving.

In [4]:
training_results = {}

for scenario in SCENARIOS:
    print(f"\n{'=' * 20} Training {scenario} {'=' * 20}")
    training_results[scenario] = train(scenario, config=configs[scenario])

training_results


==================== Training SB1 ====================
SB1 epoch   10/500: train=0.060430, validation=0.062912
SB1 epoch   20/500: train=0.039959, validation=0.041644
SB1 epoch   30/500: train=0.033087, validation=0.035159
SB1 epoch   40/500: train=0.028997, validation=0.031157
SB1 epoch   50/500: train=0.021890, validation=0.023693
SB1 epoch   60/500: train=0.016461, validation=0.017967
SB1 epoch   70/500: train=0.012815, validation=0.014357
SB1 epoch   80/500: train=0.012078, validation=0.013196
SB1 epoch   90/500: train=0.008991, validation=0.010360
SB1 epoch  100/500: train=0.008345, validation=0.009633
SB1 epoch  110/500: train=0.006282, validation=0.007478
SB1 epoch  120/500: train=0.005520, validation=0.006576
SB1 epoch  130/500: train=0.004477, validation=0.005488
SB1 epoch  140/500: train=0.003857, validation=0.004828
SB1 epoch  150/500: train=0.003324, validation=0.004208
SB1 epoch  160/500: train=0.002940, validation=0.003792
SB1 epoch  170/500: train=0.002692, validation=0

{'SB1': {'train_mse': 0.0006634121374227106,
  'validation_mse': 0.0007412322956952266},
 'SB2': {'train_mse': 0.000552361179376021,
  'validation_mse': 0.0007097522557160119},
 'BG1': {'train_mse': 0.00042605034704320133,
  'validation_mse': 0.0005449748950923095},
 'BG2': {'train_mse': 0.0004333433990832418,
  'validation_mse': 0.0005376769640861312},
 'IP1': {'train_mse': 0.21528667652606964,
  'validation_mse': 0.8061146773397923}}

## Evaluate saved models

Evaluation uses all generated samples and reports MSE, MAE, maximum absolute error, and mean relative error.

In [6]:
evaluation_results = {
    scenario: evaluate(scenario)
    for scenario in ALL_SCENARIOS
}

evaluation_results

SB1 evaluation: mse=0.00068066, mae=0.0159267, max_ae=1.32071, mean_relative_error=21286.9
SB2 evaluation: mse=0.000584968, mae=0.016379, max_ae=0.5723, mean_relative_error=11951.6
BG1 evaluation: mse=0.000449878, mae=0.0148238, max_ae=0.180091, mean_relative_error=0.0517371
BG2 evaluation: mse=0.00045411, mae=0.0147428, max_ae=0.1796, mean_relative_error=4594.88
IP1 evaluation: mse=0.334565, mae=0.394775, max_ae=8.54198, mean_relative_error=302833


{'SB1': {'mse': 0.00068066013045609,
  'mae': 0.015926746651530266,
  'max_ae': 1.320711612701416,
  'mean_relative_error': 21286.896484375},
 'SB2': {'mse': 0.0005849676672369242,
  'mae': 0.016379021108150482,
  'max_ae': 0.5723000764846802,
  'mean_relative_error': 11951.6005859375},
 'BG1': {'mse': 0.0004498781927395612,
  'mae': 0.01482375431805849,
  'max_ae': 0.18009105324745178,
  'mean_relative_error': 0.05173712596297264},
 'BG2': {'mse': 0.0004541103553492576,
  'mae': 0.014742822386324406,
  'max_ae': 0.17960044741630554,
  'mean_relative_error': 4594.8759765625},
 'IP1': {'mse': 0.3345645070075989,
  'mae': 0.39477452635765076,
  'max_ae': 8.541975021362305,
  'mean_relative_error': 302833.0}}